# **Section 5, Estimation: the bootstrap and confidence intervals**

Everything runs on its own, the data is
built in the notebook, so you can change a number and re-run.


#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- [13 Estimation](https://inferentialthinking.com/chapters/13/estimation/)
- [14.1 Properties of the Mean](https://inferentialthinking.com/chapters/14/1/properties-of-the-mean/)
- [14.2 Training and Testing](https://inferentialthinking.com/chapters/14/2/variability/)

**Where this is used.** The bootstrap and percentile machinery here is what
Lab 6 Sections 3 and 4 use to build a confidence interval for a regression
slope. Nothing in this Section has a lab of its own; it is the foundation the
Section 6 lab work stands on.


In [ ]:
# Run this cell first -- the install takes about a minute in the browser
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [The problem: one sample, unknown parameter](#1)
2. [Percentiles](#2)
3. [The bootstrap](#3)
4. [Building a confidence interval](#4)
5. [What x% confidence actually means](#5)
6. [Sample size and confidence level](#6)
7. [Using an interval as a hypothesis test](#7)
8. [Centre and spread: mean, median, SD](#8)
9. [Standard units](#9)
10. [A note on the bell curve](#10)
11. [Quick reference](#11)


---

<a id='1'></a>
## **1. The problem: one sample, unknown parameter**

You want a **parameter**, a number about the whole population. You have one
**sample**. The statistic you compute from it is an estimate, and it would come
out differently with a different sample.

Section 3 asked *could this have happened by chance?* This Section asks something
harder: **how far from the truth is my estimate likely to be?**

The obstacle is that you cannot go back and take more samples. The bootstrap is
the trick that gets round it.


In [ ]:
# A population of salaries we can see. In real work you never can.
np.random.seed(8)
population = Table().with_column(
    'Salary', np.round(np.random.lognormal(11.3, 0.55, 12000)))
parameter = np.median(population.column('Salary'))
print('true median salary:', parameter)

In [ ]:
# One sample of 400 -- this is all you would actually have
sample = population.sample(400, with_replacement=False)
statistic = np.median(sample.column('Salary'))
print('sample median:', statistic)

---

<a id='2'></a>
## **2. Percentiles**

The **p-th percentile** is the smallest value in a sorted list that is at least
as large as p% of the values.

```
percentile(p, array)
```

`percentile(50, ...)` is the median. Percentiles are how you cut the ends off a
distribution, which is what a confidence interval does.


In [ ]:
sizes = make_array(3, 1, 9, 5, 7)
print('sorted:', np.sort(sizes))
print('50th:', percentile(50, sizes))
print('25th:', percentile(25, sizes))
print('90th:', percentile(90, sizes))

> **Not the same as `np.median`.** `percentile` always returns an element that
> is actually in the array. `np.median` averages the middle two when the count
> is even, so it can return a value no individual has.


In [ ]:
evens = make_array(1, 2, 3, 4)
print('percentile(50, evens) =', percentile(50, evens))
print('np.median(evens)      =', np.median(evens))

---

<a id='3'></a>
## **3. The bootstrap**

You cannot draw new samples from the population. But **your sample is the best
picture of the population you have**, so draw from *it* instead, with
replacement, at the same size.

That is a bootstrap resample. It mimics the act of sampling, using only what you
already hold.

```
sample.sample()                       # same size, WITH replacement -- the default
```

With replacement is essential. Without it, a resample of the same size would just
be the original sample reordered, and every statistic would be identical.


In [ ]:
one_resample = sample.sample()
print('original size:', sample.num_rows, ' resample size:', one_resample.num_rows)
print('original median:', np.median(sample.column('Salary')))
print('resample median:', np.median(one_resample.column('Salary')))

In [ ]:
# Repeat it. Each resample gives a slightly different median.
bootstrap_medians = make_array()

for i in np.arange(2000):
    resample = sample.sample()
    bootstrap_medians = np.append(bootstrap_medians,
                                  np.median(resample.column('Salary')))

Table().with_column('Bootstrap median', bootstrap_medians).hist()
plots.title('2000 bootstrap medians from one sample of 400')
plots.show()

---

<a id='4'></a>
## **4. Building a confidence interval**

Cut off the extreme 2.5% at each end of the bootstrap distribution. What is left
is a **95% confidence interval**.


In [ ]:
left = percentile(2.5, bootstrap_medians)
right = percentile(97.5, bootstrap_medians)
print(f'95% confidence interval: [{left:.0f}, {right:.0f}]')
print(f'true parameter:          {parameter:.0f}')
print('interval contains it:', left <= parameter <= right)

For a 90% interval take the 5th and 95th percentiles; for 99%, the 0.5th and
99.5th. The rule: **leave out half the missing percentage at each end.**


---

<a id='5'></a>
## **5. What x% confidence actually means**

This is the sentence people get wrong.

**Wrong:** "there is a 95% chance the parameter is in this interval."

The parameter is a fixed number. It either is in your interval or it isn't, and there is no chance about it once the interval exists.

**Right:** "the *procedure* that produced this interval captures the parameter
95% of the time."

The randomness is in the sampling, not the parameter. Below, we build 100
intervals from 100 different samples and count how many contain the truth.


In [ ]:
def one_interval(size):
    """Draw a fresh sample, bootstrap it, return a 95% interval."""
    samp = population.sample(size, with_replacement=False)
    meds = make_array()
    for i in np.arange(400):
        meds = np.append(meds, np.median(samp.sample().column('Salary')))
    return percentile(2.5, meds), percentile(97.5, meds)

intervals = [one_interval(400) for i in np.arange(100)]
covered = sum(1 for lo, hi in intervals if lo <= parameter <= hi)
print(f'{covered} of 100 intervals contain the true median')

In [ ]:
# Each horizontal line is one interval; the vertical line is the truth.
plots.figure(figsize=(7, 6))
for i, (lo, hi) in enumerate(intervals):
    hit = lo <= parameter <= hi
    plots.plot([lo, hi], [i, i], color=('C0' if hit else 'red'), lw=1)
plots.axvline(parameter, color='black', lw=2)
plots.title('100 confidence intervals; red ones miss')
plots.yticks([])
plots.show()

About five miss. That is what 95% confidence means, not a statement about any
one interval, but about how often the method works.

You never know which of your intervals is one of the red ones.


---

<a id='6'></a>
## **6. Sample size and confidence level**

Two dials, and they move the interval in different ways.

**Bigger sample → narrower interval.** More data, less uncertainty.

**Higher confidence → wider interval.** To be right more often you must claim
less precisely.


In [ ]:
def interval_width(size, level):
    """Width of a bootstrap interval at a given sample size and confidence level."""
    samp = population.sample(size, with_replacement=False)
    meds = make_array()
    for i in np.arange(400):
        meds = np.append(meds, np.median(samp.sample().column('Salary')))
    tail = (100 - level) / 2
    return percentile(100 - tail, meds) - percentile(tail, meds)

for n in [100, 400, 1600]:
    print(f'n = {n:>5}, 95% interval width: {interval_width(n, 95):.0f}')

In [ ]:
for level in [80, 95, 99]:
    print(f'{level}% interval width at n=400: {interval_width(400, level):.0f}')

Quadrupling the sample roughly halves the width. That relationship, width
scaling with $1/\sqrt{n}$, is why doubling your data helps less than you might
hope.


---

<a id='7'></a>
## **7. Using an interval as a hypothesis test**

A confidence interval can answer a testing question directly.

**If a proposed value falls outside a 95% interval, reject it at the 5% level.**
If it falls inside, you cannot.

That is the same conclusion a hypothesis test would reach, from the same data, 
and it also tells you the plausible range, which a p-value does not.


In [ ]:
proposed = 70000
print(f'95% interval: [{left:.0f}, {right:.0f}]')
print(f'is {proposed} inside?', left <= proposed <= right)
print('->', 'cannot reject' if left <= proposed <= right else 'reject at the 5% level')

---

<a id='8'></a>
## **8. Centre and spread: mean, median, SD**

The **mean** balances the distribution; the **median** splits it in half. They
agree when the distribution is symmetric and part company when it is skewed.


In [ ]:
skewed = population.column('Salary')
print(f'mean:   {np.mean(skewed):,.0f}')
print(f'median: {np.median(skewed):,.0f}')
population.hist('Salary', bins=np.arange(0, 400000, 15000))
plots.title('Salaries: a right-skewed distribution')
plots.show()

The mean sits to the right of the median, dragged by the long tail. **A few very
large values move the mean and barely touch the median**, which is why the
median is the usual choice for incomes.

The **standard deviation** measures spread: roughly, how far a typical value
sits from the mean.

$$\text{SD} = \sqrt{\text{mean of }(\text{value} - \text{mean})^2}$$


In [ ]:
values = make_array(2, 3, 3, 9)
deviations = values - np.mean(values)
print('values:     ', values)
print('deviations: ', deviations)
print('squared:    ', deviations ** 2)
print('SD by hand: ', np.sqrt(np.mean(deviations ** 2)))
print('np.std:     ', np.std(values))

> **Why square?** The deviations always sum to zero, the positives cancel the
> negatives. Squaring removes the signs; the square root at the end returns the
> answer to the original units.


---

<a id='9'></a>
## **9. Standard units**

Converting to standard units asks: **how many SDs above or below the mean is
this value?**

$$z = \frac{\text{value} - \text{mean}}{\text{SD}}$$

It strips the units away, so quantities measured on completely different scales
become comparable. Chapter 15's correlation is defined entirely in these terms.


In [ ]:
def standard_units(x):
    """Converts an array to standard units."""
    return (x - np.mean(x)) / np.std(x)

heights = make_array(150, 165, 172, 180, 195)
weights = make_array(52, 63, 70, 84, 99)

print('heights in SU:', np.round(standard_units(heights), 2))
print('weights in SU:', np.round(standard_units(weights), 2))

In [ ]:
# In standard units the mean is 0 and the SD is 1, whatever you started with
su = standard_units(heights)
print('mean:', round(np.mean(su), 10))
print('SD:  ', round(np.std(su), 10))

---

<a id='10'></a>
## **10. A note on the bell curve**

You will meet a shortcut elsewhere that this course does not use.

Many distributions are roughly **bell-shaped**, and for those there is a rule of
thumb: about 68% of values lie within one SD of the mean, about 95% within two,
and about 99.7% within three. People use it to build confidence intervals
without simulating anything, take the estimate, add and subtract two SDs, done.

**We bootstrap instead**, for one reason: the shortcut only works when the
distribution is bell-shaped, and resampling works whatever the shape. The salary
distribution above is nowhere near bell-shaped, and its bootstrap interval is
perfectly valid.

You should recognise "±2 SD" when you see it in a paper or a colleague's chart.
You do not need it here.


---
## **Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** `percentile(50, arr)` and `np.median(arr)` disagree. How is that possible?

<details>
<summary><strong>Answer</strong></summary>

When the count is <strong>even</strong>. <code>percentile</code> always returns an element that is actually in the array; <code>np.median</code> averages the middle two, which may be a value the data never took.

</details>

**Q2.** A 95% confidence interval for a mean is [12, 18]. Why is it wrong to say there is a 95% chance the true mean is between 12 and 18?

<details>
<summary><strong>Answer</strong></summary>

The true mean is a fixed number: it either is in that interval or it is not. The 95% describes the <strong>procedure</strong>: if you repeated the whole sampling and interval-building many times, about 95% of the intervals produced would cover the true value. This one either does or does not.

</details>

**Q3.** How large should a bootstrap resample be, and why is that the answer?

<details>
<summary><strong>Answer</strong></summary>

The <strong>same size as the original sample</strong>, drawn with replacement. The bootstrap imitates drawing a fresh sample from the population, so it has to imitate the sample size too. A smaller resample would overstate the variability.

</details>

**Q4.** Your 95% interval for a difference is [1.4, 6.2]. What does that tell you about the hypothesis that the difference is zero?

<details>
<summary><strong>Answer</strong></summary>

Zero is <strong>outside</strong> the interval, so at the 5% level you reject the hypothesis that the difference is zero. An interval used this way is a hypothesis test: check whether the null value falls inside.

</details>

**Q5.** You want a narrower interval. Does raising the confidence level from 95% to 99% help?

<details>
<summary><strong>Answer</strong></summary>

No, it does the opposite. A higher confidence level needs a <strong>wider</strong> interval, because it must catch more of the bootstrap distribution. To narrow an interval you need a <strong>larger sample</strong>, not more confidence.

</details>

**Q6.** What are the mean and SD of any array after conversion to standard units, and why is that useful?

<details>
<summary><strong>Answer</strong></summary>

Mean <strong>0</strong> and SD <strong>1</strong>, always. That is what makes two quantities on completely different scales comparable, and it is why the correlation coefficient works: it is the average product of two variables in standard units.

</details>

---

<a id='11'></a>
## **11. Quick reference**

### **The bootstrap, in five lines**

```python
stats = make_array()
for i in np.arange(2000):
    resample = sample.sample()              # same size, WITH replacement
    stats = np.append(stats, np.median(resample.column('X')))
percentile(2.5, stats), percentile(97.5, stats)
```

### **Percentiles for a confidence level**

| Level | Take |
|---|---|
| 80% | 10th and 90th |
| 90% | 5th and 95th |
| 95% | 2.5th and 97.5th |
| 99% | 0.5th and 99.5th |

### **Calls**

| Call | Gives |
|---|---|
| `percentile(p, array)` | the p-th percentile, always an actual element |
| `tbl.sample()` | a bootstrap resample: same size, with replacement |
| `np.mean` · `np.median` | centre |
| `np.std(array)` | standard deviation |
| `(x - np.mean(x)) / np.std(x)` | standard units |

### **Things that catch people out**

| | |
|---|---|
| Bootstrapping without replacement | every resample is the original reordered |
| "95% chance the parameter is in here" | the parameter is fixed; the *method* works 95% of the time |
| `percentile(50, ...)` vs `np.median` | they differ when the count is even |
| Bigger sample | narrower interval |
| Higher confidence | **wider** interval |
| Skewed data | the mean moves, the median holds |

---

### **Textbook**

- [Chapter 13, Estimation](https://inferentialthinking.com/chapters/13/Estimation.html)
- [Chapter 13.2, Bootstrap](https://inferentialthinking.com/chapters/13/2/Bootstrap.html)
- [Chapter 13.3, Confidence intervals](https://inferentialthinking.com/chapters/13/3/Confidence_Intervals.html)
- [Chapter 14.1, Properties of the mean](https://inferentialthinking.com/chapters/14/1/Properties_of_the_Mean.html)
- [Chapter 14.2, Variability](https://inferentialthinking.com/chapters/14/2/Variability.html)
